# Class-Balanced T1 Training and Validation

This is the Colab training notebook. Upload `medgemma-secondary-kaggle.tar` and the updated `scripts/train_t1_qlora.py` from the repository. The archive contains the full training and validation images. A four-record smoke test runs before the expensive full pass; the notebook never loads the test split.

In [ ]:
%pip install -q -U "transformers>=4.50.0" peft accelerate bitsandbytes pillow scikit-learn safetensors
import torch
print("torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Select a CUDA GPU runtime before continuing")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from google.colab import files
from pathlib import Path
import shutil
import tarfile

print("Select medgemma-secondary-kaggle.tar from its folder.")
uploaded_archive = files.upload()
archive_name = next(name for name in uploaded_archive if name.endswith(".tar"))
print("Now select train_t1_qlora.py from its folder.")
uploaded_trainer = files.upload()
trainer_name = next(name for name in uploaded_trainer if name.endswith("train_t1_qlora.py"))
ROOT = Path("/content/medgemma-secondary")
ROOT.mkdir(exist_ok=True)
with tarfile.open(Path("/content") / archive_name, "r") as archive:
    archive.extractall(ROOT)
trainer_path = ROOT / "scripts/train_t1_qlora.py"
shutil.copy2(Path("/content") / trainer_name, trainer_path)
assert "--class-balanced" in trainer_path.read_text()
required = [
    ROOT / "configs/secondary.json",
    ROOT / "data/secondary/instruction_tuning_mixed/train.jsonl",
    ROOT / "data/secondary/instruction_tuning_mixed/val.jsonl",
    ROOT / "data/secondary/processed/microscopy/tryp",
]
assert all(path.exists() for path in required), required
print("Using root:", ROOT)
print("Updated trainer:", trainer_path)

In [ ]:
import getpass
import os

if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass.getpass("Paste HF_TOKEN: " )
assert os.environ["HF_TOKEN"]
print("HF_TOKEN loaded")

In [ ]:
import subprocess
import sys

OUTPUT_DIR = Path("/content/medgemma-t1-qlora-balanced")
command = [
    sys.executable, "-u", str(trainer_path),
    "--root", str(ROOT),
    "--output", str(OUTPUT_DIR),
    "--epochs", "1",
    "--learning-rate", "2e-4",
    "--gradient-accumulation-steps", "8",
    "--seed", "42",
    "--class-balanced",
    "--skip-eval",
]
smoke_command = command + ["--max-train", "4"]
print("Running four-record smoke test before full training...", flush=True)
subprocess.run(smoke_command, check=True)
assert (OUTPUT_DIR / "adapter_model.safetensors").exists()
shutil.rmtree(OUTPUT_DIR)
print("Smoke test passed; starting full class-balanced training...", flush=True)
subprocess.run(command, check=True)
assert (OUTPUT_DIR / "adapter_model.safetensors").exists()
print("Balanced adapter saved:", OUTPUT_DIR)

In [ ]:
import json

command = [
    sys.executable, "-u", str(trainer_path),
    "--root", str(ROOT),
    "--output", str(OUTPUT_DIR),
    "--eval-only",
    "--max-val", "488",
    "--max-new-tokens", "8",
]
subprocess.run(command, check=True)
metrics = json.loads((OUTPUT_DIR / "t1_metrics.json").read_text())
print(json.dumps(metrics, indent=2))
summary = json.loads((OUTPUT_DIR / "training_summary.json").read_text())
print(json.dumps(summary, indent=2))
assert metrics["records"] == 488
assert metrics["invalid_outputs"] == 0
print("Validation gate: inspect negative_control_recall and macro_f1 before testing.")

In [ ]:
from google.colab import files

adapter_zip = shutil.make_archive("/content/medgemma-t1-qlora-balanced", "zip", root_dir=OUTPUT_DIR.parent, base_dir=OUTPUT_DIR.name)
print("Download this only if validation passes the gate:", adapter_zip)
files.download(adapter_zip)